In [1]:
import json
import pickle
import pandas as pd
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score, f1_score

In [2]:
def calculate_all_metrics(y_true, y_pred, y_proba):
    
    return {
        "test_precision": precision_score(y_true, y_pred),
        "test_recall": recall_score(y_true, y_pred),
        "test_roc_auc": roc_auc_score(y_true, y_proba),
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_f1": f1_score(y_true, y_pred)
    }

In [3]:

model_ids = ["model_1", "model_2", "model_1a", "model_2a", "model_1b", "model_2b"]
model_types = ["xgboost", "rf"]
all_results = []
for m_id in model_ids:
    X = pd.read_csv(f"data/X_test_{m_id}.csv", keep_default_na=False)
    y = pd.read_csv(f"data/y_test_{m_id}.csv", keep_default_na=False).values.ravel()
    for m_type in model_types:
        with open(f"models/{m_id}_{m_type}.pkl", "rb") as f:
            loaded_model = pickle.load(f)
        y_preds = loaded_model.predict(X)
        y_pred_probs = loaded_model.predict_proba(X)[:, 1]
        metrics = {
                "model_number": m_id,
                "model_type": m_type
            }
        metrics.update(calculate_all_metrics(y, y_preds, y_pred_probs))
        all_results.append(metrics)

pd.DataFrame(all_results).to_csv("results/final_model_test_metrics.csv", index=False)